# Notebook 07 - Time-Frequency Trade-off

## Goal
Compare short and long windows to understand time vs frequency resolution.


## Agenda
- Compute short-window spectrogram
- Compute long-window spectrogram
- Compare visual sharpness
- Document trade-off


## Concept and Math

Short windows improve time localization but reduce frequency precision.
Long windows improve frequency precision but smear events in time.
STFT design is always a trade-off.


In [ ]:
from pathlib import Path
import numpy as np
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))
if not audio_files:
    raise FileNotFoundError("No .flac or .wav found under ../dataset")

audio_path = audio_files[0]
print(f"Using: {audio_path}")

wave, sr = lb.load(audio_path, sr=16000, mono=True)
configs = [(256, 80, "Short"), (1024, 320, "Long")]

plt.figure(figsize=(11, 6))
for i, (n_fft, hop, title) in enumerate(configs, start=1):
    S = lb.stft(wave, n_fft=n_fft, hop_length=hop, win_length=n_fft)
    D = lb.amplitude_to_db(np.abs(S), ref=np.max)
    plt.subplot(2, 1, i)
    librosa.display.specshow(D, sr=sr, hop_length=hop, x_axis="time", y_axis="hz")
    plt.title(f"{title} window")
plt.tight_layout()
plt.show()


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torchaudio
import torch

wave_t, sr_t = torchaudio.load(str(audio_path))
wave_t = wave_t.mean(dim=0)
S_short = torch.stft(wave_t, n_fft=256, hop_length=80, return_complex=True)
S_long = torch.stft(wave_t, n_fft=1024, hop_length=320, return_complex=True)
print(S_short.shape, S_long.shape)


## Review Checklist
- Which window captures transients better?
- Which window separates close frequencies better?
- How would you pick window size for your dataset?
